# Análises de Zika — esquema estrela unificado

Execute após **um** dos pipelines. Ambos publicam o mesmo esquema MySQL.
As consultas e visualizações ficam neste arquivo, separadas da engenharia dos pipelines.
2026 é parcial; contagens descrevem notificações, não incidência nem risco populacional.
Gráficos são exportados para `saidas/analises/`.

In [ ]:
from pathlib import Path
import sys

# Execute na raiz do clone ou em notebooks/ (inclusive no Colab).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pipelines/contrato.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Clone o repositório e abra o notebook dentro dele.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
# Crie previamente um banco dedicado ao projeto, usando utf8mb4.
# Configure a conexão antes de executar. A senha vem do ambiente, sem ser salva no notebook.
os.environ.setdefault('MYSQL_HOST', '127.0.0.1')
os.environ.setdefault('MYSQL_PORT', '3306')
os.environ.setdefault('MYSQL_USER', 'root')
os.environ.setdefault('MYSQL_DATABASE', 'zika_dw')
# os.environ['MYSQL_PASSWORD'] = getpass.getpass('Senha MySQL: ')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pipelines.database import connect
output = ROOT / 'saidas/analises'
output.mkdir(parents=True, exist_ok=True)

def query(sql):
    with connect() as connection, connection.cursor() as cursor:
        cursor.execute(sql)
        return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)


## 1. Notificações mensais e sazonalidade

In [ ]:
monthly = query("""SELECT t.ano, t.mes, CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes
FROM fato_notificacao_zika f JOIN dim_tempo t ON t.sk_tempo=f.fk_tempo_notificacao
GROUP BY t.ano,t.mes ORDER BY t.ano,t.mes""")
ax = monthly.pivot(index='mes', columns='ano', values='notificacoes').plot(marker='o')
ax.set(xlabel='Mês', ylabel='Notificações', title='Série mensal (2026 parcial)')
plt.tight_layout()
plt.savefig(output / '01_sazonalidade.png')
plt.show()
display(monthly)

## 2. Gestantes por UF de residência

In [ ]:
pregnant = query("""SELECT l.sigla_uf, CAST(SUM(f.fl_gestante) AS SIGNED) AS gestantes,
CAST(SUM(f.fl_gestante * f.fl_confirmado) AS SIGNED) AS confirmadas
FROM fato_notificacao_zika f JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
GROUP BY l.sigla_uf ORDER BY gestantes DESC""")
ax = pregnant.set_index('sigla_uf')[['gestantes','confirmadas']].plot.bar(figsize=(12,4))
ax.set(xlabel='UF de residência (NI = não informada)', ylabel='Notificações', title='Gestantes notificadas e confirmadas')
plt.tight_layout()
plt.savefig(output / '02_gestantes.png')
plt.show()
display(pregnant)

## 3. Tempo até notificação e encerramento

In [ ]:
delay = query("""SELECT l.regiao_brasil, AVG(f.qt_dias_atraso_notif) AS atraso_notificacao,
AVG(f.qt_dias_encerramento) AS tempo_encerramento
FROM fato_notificacao_zika f JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
GROUP BY l.regiao_brasil ORDER BY l.regiao_brasil""")
ax = delay.set_index('regiao_brasil').astype(float).plot.barh()
ax.set(xlabel='Dias (média dos intervalos válidos de 0 a 365)', ylabel='Região', title='Tempos da vigilância')
plt.tight_layout()
plt.savefig(output / '03_tempos.png')
plt.show()
display(delay)